# Notebook For Analysis

# Setup

In [1]:
!echo $HOSTNAME
!python --version
!nvidia-smi

g003
Python 3.12.5
Sun Jun 21 19:21:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.08              Driver Version: 575.57.08      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-PCIE-40GB          Off |   00000000:CA:00.0 Off |                    0 |
| N/A   26C    P0             36W /  250W |     423MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+----------------------------

In [ ]:
# General Libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import einops
from fancy_einsum import einsum
import os
import tqdm.auto as tqdm
import random
from pathlib import Path
import plotly.express as px
import copy

from typing import List, Union, Optional
from functools import partial
import itertools
from IPython.display import HTML

# CSV Use Libraries
import pandas as pd

# Function Imports
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Unused
"""
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoConfig, AutoTokenizer
import dataclasses
import datasets
import ast
from transformers import get_cosine_schedule_with_warmup #Used for lr?
from torch.nn.utils import clip_grad_norm_
"""

### Save Directories
- Specify Notebook Directory
- Specify Dataset Directory

In [ ]:
import sys

# Add the Catagorical Transformer directory to the path so config.py can be imported
_ct_dir = str(Path(os.getcwd()) / "Catagorical Transformer")
if _ct_dir not in sys.path:
    sys.path.insert(0, _ct_dir)

from config import *

# DATA_PATH and PTH_LOCATION are now resolved from config.py
print(f"Data Dir:  {DATA_PATH}")
print(f"Model PTH: {PTH_LOCATION}")

In [4]:
# GPU Memory Setup

# Limit GPU usage to 90% of total memory
torch.cuda.set_per_process_memory_fraction(0.9, device=0)

# Prevent fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128,expandable_segments:True"

# Clear leftover memory
torch.cuda.empty_cache()

print("Allocated memory:", torch.cuda.memory_allocated() / 1024**3, "GB")
print("Cached memory:   ", torch.cuda.memory_reserved() / 1024**3, "GB")

Allocated memory: 0.0 GB
Cached memory:    0.0 GB


## Define Graphing Functs

In [5]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"
print(f"Using renderer: {pio.renderers.default}")

Using renderer: notebook_connected


In [6]:
pio.templates['plotly'].layout.xaxis.title.font.size = 20
pio.templates['plotly'].layout.yaxis.title.font.size = 20
pio.templates['plotly'].layout.title.font.size = 30

In [ ]:
import sys
import transformer_lens
import transformer_lens.utilities as utils
from transformer_lens import HookedTransformer
import transformer_lens.config.hooked_transformer_config as htc
sys.modules['transformer_lens.HookedTransformerConfig'] = sys.modules['transformer_lens.config.hooked_transformer_config']

In [ ]:
# Unused plotting functions in favor of Neel Plotly
"""
def imshow(tensor, renderer=None, xaxis="", yaxis="", **kwargs):
    px.imshow(utils.to_numpy(tensor), color_continuous_midpoint=0.0, color_continuous_scale="RdBu", labels={"x":xaxis, "y":yaxis}, **kwargs).show(renderer)

def line(tensor, renderer=None, xaxis="", yaxis="", **kwargs):
    px.line(utils.to_numpy(tensor), labels={"x":xaxis, "y":yaxis}, **kwargs).show(renderer)

def scatter(x, y, xaxis="", yaxis="", caxis="", renderer=None, **kwargs):
    x = utils.to_numpy(x)
    y = utils.to_numpy(y)
    px.scatter(y=y, x=x, labels={"x":xaxis, "y":yaxis, "color":caxis}, **kwargs).show(renderer)
"""

## Set Paths and GPU Devices

In [ ]:
# PTH_LOCATION is imported from config.py; create the directory if it doesn't exist yet
os.makedirs(Path(PTH_LOCATION).parent, exist_ok=True)

In [ ]:
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device count: {torch.cuda.device_count()}")

for i in range(torch.cuda.device_count()):
    print(f"Device {i}: {torch.cuda.get_device_name(i)}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device Name: {device}")

device1 = None
if torch.cuda.is_available():
    device1 = torch.device("cuda:0")

## Define Model and Optimizer Params

**Task:** Coxeter Group input and Binary Classification Output

- `[B,T,C]`
    - Batch Size | dataset size fed into model
    - Sequence Length | ctx
    - Number of Features | d_model

- Sequence (an input):
    - sequence: `[1,2,3,4]`
    - the label: `0` or `1`
    - each token has a `d_model` lengthed "features" vector

In [ ]:
torch.manual_seed(seed=DATA_SEED)
torch.cuda.manual_seed_all(DATA_SEED)

# Architecture config is overridden by the saved checkpoint below,
# but keeping it here documents the intended architecture.
cfg = htc.HookedTransformerConfig(
    n_ctx=SEQUENCE_LENGTH,
    n_layers=LAYERS,
    n_heads=HEADS,
    d_head=DIM_HEADS,
    d_model=DIM_MODEL,
    d_mlp=DIM_MLP,
    d_vocab=TOKEN_TYPES,
    d_vocab_out=DIM_OUTPUT,
    act_fn=TYPE,
    init_weights=INIT_WEIGHTS,
    device=device,
    n_devices=NUM_DEVICES,
    seed=LENS_SEED,
    attention_dir=ATTENTION_DIRECTION,
    normalization_type=NORMALIZATION,
    positional_embedding_type=POSITIONAL_EMBEDDING_TYPE,
)

# Load config and weights from checkpoint
cached_data = torch.load(PTH_LOCATION, weights_only=False)
cfg = cached_data["config"]

# --- Optimizer Config ---
lr       = LEARNING_RATE
wd       = WEIGHT_DECAY
betas    = BETAS
patience = PATIENCE

# --- Model, Optimizer, Scheduler ---
model     = HookedTransformer(cfg)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd, betas=betas)
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=patience, factor=0.5)

# NOTE: warmup scheduler causes ~50/50 accuracy — don't initialize unless using mini-batches
# scheduler = get_cosine_schedule_with_warmup(optimizer, ...)

# --- Load State Dicts ---
model.load_state_dict(cached_data["model"])
optimizer.load_state_dict(cached_data["optimizer"])
scheduler.load_state_dict(cached_data["scheduler"])
model_checkpoints = cached_data["checkpoints"]
checkpoint_epochs = cached_data["checkpoint_epochs"]
test_losses       = cached_data["test_losses"]
train_losses      = cached_data["train_losses"]
train_accuracies  = cached_data["train_accuracies"]
test_accuracies   = cached_data["test_accuracies"]

# Disable biases for interpretability
for name, param in model.named_parameters():
    if "b_" in name:
        param.requires_grad = False

## Define Loss and Accuracy Functions

In [ ]:
torch.cuda.empty_cache()
clsDex = 0  # index of CLS token, kept for attention pattern analysis

_bce = torch.nn.BCEWithLogitsLoss(reduction='none')

def descent_loss_fn(logits, targets, mask):
    """Masked multi-label BCE: per generator, is it a right descent of this prefix?
    logits/targets: [batch, seq_len, n_generators]; mask: [batch, seq_len]."""
    per_unit = _bce(logits, targets)
    m = mask.unsqueeze(-1).float()
    return (per_unit * m).sum() / (m.sum() * logits.size(-1))

def descent_accuracy_fn(logits, targets, mask):
    """Per-prefix exact-set-match accuracy over non-pad positions (scalar)."""
    preds = (logits > 0).float()
    correct_bits = (preds == targets).float().sum(dim=-1)
    exact = (correct_bits == logits.size(-1)).float()
    m = mask.float()
    return (exact * m).sum() / m.sum()

def descent_sequence_accuracy(logits, targets, mask):
    """Per-sequence fraction of prefixes with an exactly-correct descent set.
    Returns shape (batch_size,)."""
    preds = (logits > 0).float()
    correct_bits = (preds == targets).float().sum(dim=-1)
    exact = (correct_bits == logits.size(-1)).float()
    m = mask.float()
    return (exact * m).sum(dim=-1) / m.sum(dim=-1).clamp(min=1)

## Define Attention Masking Functions

In [ ]:
# Create attention mask: 1 for real tokens, 0 for padding
def create_attention_mask(data_tensor):
    # assuming padding is exactly 0
    return (data_tensor != 0).int()

def pad_mask_hook(attn_scores, hook, mask):
    # attn_scores: [batch, head, q_pos, k_pos]
    # mask: [batch, seq_len]
    pad_mask = mask.unsqueeze(1).unsqueeze(2)  # [batch, 1, 1, seq_len]
    attn_scores = attn_scores.masked_fill(~pad_mask.bool(), float('-inf'))
    return attn_scores

def register_pad_mask_hook(model, attention_mask):
    def mask_hook(attn_scores, hook):
        return pad_mask_hook(attn_scores, hook, attention_mask)
    for layer in range(cfg.n_layers):
        model.blocks[layer].attn.hook_attn_scores.add_hook(mask_hook)

## Initialize Datasets

In [ ]:
import ast

def load_descent_dataset(csv_path, n_generators):
    """
    Loads the two-column descent CSV (`word`, `descents`) and returns:
      tokens  : [N, seq_len]               padded generator IDs (0 = padding)
      targets : [N, seq_len, n_generators] multi-hot right-descent set per prefix
      mask    : [N, seq_len]               1 for real letters, 0 for padding
    The `descents` column stores one bitmask int per position (bit j <=> generator
    j+1), with -1 on padding; we decode it into the multi-hot target.
    """
    df = pd.read_csv(csv_path)
    words = [[int(x) for x in ast.literal_eval(w)] for w in df["word"]]
    descs = [[int(x) for x in ast.literal_eval(d)] for d in df["descents"]]

    tokens = torch.tensor(words, dtype=torch.long)
    mask = (tokens != 0).long()

    bitmasks = torch.tensor(descs, dtype=torch.long)
    bits = torch.arange(n_generators)
    targets = ((bitmasks.clamp(min=0).unsqueeze(-1) >> bits) & 1).float()
    targets = targets * mask.unsqueeze(-1).float()

    print(f"Loaded dataset: tokens {tuple(tokens.shape)} | targets {tuple(targets.shape)}")
    return tokens, targets, mask

In [ ]:
# Load the single dataset and apply the same seed+split used during training
all_tokens, all_targets, all_mask = load_descent_dataset(DATA_PATH / DATA_CSV, DIM_OUTPUT)

torch.manual_seed(DATA_SEED)
perm        = torch.randperm(all_tokens.size(0))
all_tokens  = all_tokens[perm]
all_targets = all_targets[perm]
all_mask    = all_mask[perm]

n_train = int(TRAINING_SPLIT * all_tokens.size(0))
train_tokens,  test_tokens  = all_tokens[:n_train],  all_tokens[n_train:]
train_targets, test_targets = all_targets[:n_train], all_targets[n_train:]
train_mask,    test_mask    = all_mask[:n_train],    all_mask[n_train:]

print(f"Train size: {train_tokens.shape[0]} | Test size: {test_tokens.shape[0]} | Seq Len: {train_tokens.shape[1]}")

In [ ]:
train_tokens,  test_tokens  = train_tokens.to(device1),  test_tokens.to(device1)
train_targets, test_targets = train_targets.to(device1), test_targets.to(device1)
train_mask,    test_mask    = train_mask.to(device1),    test_mask.to(device1)

# The padding mask used by the attention hook is exactly the non-pad indicator.
train_attention_mask = train_mask
test_attention_mask  = test_mask

# Graph Results

In [ ]:
from neel_plotly.plot import line_or_scatter, line as neel_line

In [ ]:
skipBy = 100

def createGraph(yTrain, yTest, yName, title):
    fig = line_or_scatter(
        [yTrain[::skipBy], yTest[::skipBy]],
        x=np.arange(0, len(yTrain), skipBy),
        xaxis="Epoch",
        yaxis=yName,
        log_y=True,
        title=title,
        line_labels=['train', 'test'],
        toggle_x=True,
        toggle_y=True,
        plot_type="line",
        return_fig=True
    )
    return fig

fig1 = createGraph(train_losses, test_losses, "Loss", "Loss Curve for Word Problem")
fig2 = createGraph(train_accuracies, test_accuracies, "Accuracy", "Accuracy Curve for Word Problem")

fig1.show()
fig2.show()

fig1.write_html("loss_curve.html")
fig2.write_html("accuracy_curve.html")

# Analysing the Model

Helpful Memory Probing Functions:

- nvidia-smi
- torch.cuda.empty_cache()
- torch.set_grad_enabled(mode=False)
- gc.collect()
- print(torch.cuda.memory_summary())
- torch profiler also (saved image on Aug 03, 2025 ~6pm)

### Helper Functions
- logit return function
- prediction function

In [ ]:
import gc

def getLogits(model: HookedTransformer, tokens, getCache: bool = False):
    """
    Applies the padding mask and runs a forward pass.
    Returns detached logits (and optionally the activation cache).
    """
    with torch.inference_mode():
        model.reset_hooks()
        mask = create_attention_mask(tokens).to(device1)
        register_pad_mask_hook(model, mask)
        if getCache:
            original_logits_og, cache = model.run_with_cache(tokens)
        else:
            original_logits_og = model(tokens)
    original_logits = original_logits_og.detach().clone()
    del original_logits_og, mask
    torch.cuda.empty_cache()
    if getCache:
        return original_logits, cache
    else:
        return original_logits

def getPredictions(model: HookedTransformer, tokens, targets, mask):
    """
    Per-sequence descent accuracy: fraction of prefixes whose descent set is
    exactly correct. Shape: (batch_size,)
    """
    logits  = getLogits(model, tokens, getCache=False)
    seq_acc = descent_sequence_accuracy(logits, targets, mask)
    del logits
    return seq_acc

def imshow(tensor, renderer=None, xaxis="", yaxis="", xlabels=None, ylabels=None, aspect="auto", **kwargs):
    fig = px.imshow(
        utils.to_numpy(tensor),
        color_continuous_midpoint=0.0,
        color_continuous_scale="RdBu",
        labels={"x": xaxis, "y": yaxis},
        aspect=aspect,
        **kwargs
    )
    if xlabels is not None:
        fig.update_xaxes(tickmode='array', tickvals=list(range(len(xlabels))), ticktext=xlabels)
    if ylabels is not None:
        fig.update_yaxes(tickmode='array', tickvals=list(range(len(ylabels))), ticktext=ylabels)
    fig.update_yaxes(scaleanchor=None)
    fig.show(renderer)
    return fig

def cleanup():
    gc.collect()
    torch.cuda.empty_cache()

## Quick Analysis + Memory Cleanup

In [ ]:
torch.set_grad_enabled(False)
gc.collect()
torch.cuda.empty_cache()

In [ ]:
original_logits, cache = getLogits(model, train_tokens, getCache=True)

Get key weight matrices:

In [ ]:
W_E = model.embed.W_E[:-1]
print("W_E", W_E.shape)
W_neur = W_E @ model.blocks[0].attn.W_V @ model.blocks[0].attn.W_O @ model.blocks[0].mlp.W_in
print("W_neur", W_neur.shape)
W_logit = model.blocks[0].mlp.W_out @ model.unembed.W_U
print("W_logit", W_logit.shape)

In [ ]:
original_loss = descent_loss_fn(original_logits, train_targets, train_mask).item()
print("Original Loss:", original_loss)

### Looking at Activations

Get all shapes:

In [ ]:
for param_name, param in cache.items():
    print(param_name, param.shape)

In [ ]:
# debug: prints the first 4 input train data values
train_tokens[:4]
print(model.W_E.shape)

## Attention Heads (average)

### Average attention over all words for each head
Note: all train data input words used and averaged out for these graphs
- x: letters (keys attended to)
- y: letters (query giving attention to x axis)

In [ ]:
n_heads = model.cfg.n_heads
seq_len = model.cfg.n_ctx
str_tokens = [str(i) for i in range(seq_len)]

with torch.inference_mode():
    for layer in range(model.cfg.n_layers):
        head_sums  = None   # reset per layer
        num_samples = 0

        for wordIndex in range(len(train_tokens)):
            # Grab attention pattern: [n_heads, seq_len, seq_len]
            attn_patterns = cache["pattern", layer][wordIndex]

            if head_sums is None:
                head_sums = torch.zeros_like(attn_patterns)

            head_sums   += attn_patterns
            num_samples += 1

        # Average attention
        avg_attn = head_sums / num_samples

        for head in range(n_heads):
            imshow(
                avg_attn[head].detach().cpu(),
                x=str_tokens,
                y=str_tokens,
                xaxis="Key (Attended To)",
                yaxis="Query (Paying Attention)",
                title=f"Layer {layer} Head {head} — Average Attention Across Dataset",
                aspect=None,
            )

### Average Attention Pattern over Attention Heads

In [ ]:
for layer in range(model.cfg.n_layers):
    attention = cache["pattern", layer].mean(dim=0)[:, clsDex, :]
    imshow(
        attention,
        title=f"Average Attention Paid | for token {clsDex} | per head | layer {layer}",
        xaxis="Source",
        yaxis="Head",
        x=[str(i) for i in range(train_tokens.shape[1])],
        ylabels=[f"{i}" for i in range(attention.shape[0])]
    )

## Attention Heads (word: X)

In [ ]:
# Look at a specific word in the dataset by its (shuffled) index
wordIndex = 5

print(f"Word: {train_tokens[wordIndex].tolist()}")

In [ ]:
# Evaluate the model on a single word
wordTensor  = train_tokens[wordIndex]
wordTokens  = wordTensor.unsqueeze(0)                  # add batch dimension
wordTargets = train_targets[wordIndex].unsqueeze(0)
wordMask    = train_mask[wordIndex].unsqueeze(0)
seq_acc = getPredictions(model, wordTokens, wordTargets, wordMask)

print(f"Word:    {wordTensor.tolist()}")
print(f"Seq Acc: {seq_acc.item():.4f}  (fraction of prefixes with an exactly-correct descent set)")

### Attention patterns per each head on ONE word
- prints 4 graphs

In [ ]:
# Prints 1 graph per attention head
# x: letters (keys attended to)
# y: letters (query giving attention to x axis)


# Move to test device, because it has more memory
chosenWord = train_tokens[wordIndex].unsqueeze(0).to(device1)  # shape (1, 22)
print(chosenWord)


# Generate labeled tokens with position

# ex: word len 22: str_tokens = [0,...,21]
str_tokens = [f"{i}" for i, tok in enumerate(chosenWord[0])]

# Visualize
n_heads = model.cfg.n_heads
n_layers = model.cfg.n_layers
for layer in range(n_layers):
    for head in range(n_heads):
        # gets the attention pattern for layer 0, pick 1 word from the batch and look at 1 head for it
        # detatch tensor from computation graph (no gradients), move tensor to cpu to make imshow heatmap
        attn_single_head = cache["pattern", layer][wordIndex, head].detach().cpu()
        imshow(
            attn_single_head,
            x=str_tokens,
            y=str_tokens,
            xaxis="Key (Attended To)",
            yaxis="Query (Paying Attention)",
            title=f"Layer {layer} Head {head} Attention Pattern (Positional Tokens)",
            aspect=None
        )


### Attention Pattern for wordIndex over attention Heads

In [ ]:
# Attention pattern for a specific word (x: generators, y: attention heads)
# Assuming train_tokens is shape (batch, seq_len), and you're using batch 0?

token_strs = [str(x) for x in chosenWord.squeeze(0).tolist()]

for layer in range(model.cfg.n_layers):
    attention = cache["pattern", layer][wordIndex][:, clsDex, :]
    imshow(
        attention,
        title=f"Attention Pattern (for word {wordIndex}) over Attention Heads in layer {layer}",
        xaxis="Source",
        yaxis="Head",
        #x=[str(i) for i in range(train_tokens.shape[1])],
        xlabels=token_strs,
        ylabels=[f"{i}" for i in range(attention.shape[0])]
    )

print([str(i) for i in range(train_tokens.shape[1])])
print(token_strs)

## Misclassification Graphs
- Experimenting with looking at length of consistently incorrect words

### Train Misclassification Graphs:

In [ ]:
from collections import defaultdict
import matplotlib.pyplot as plt

words = []
seq_accuracies = []
lengths = []
dataset_indices = []

total_by_length         = defaultdict(int)
misclassified_by_length = defaultdict(int)

# Per-sequence descent accuracy for all train examples (1.0 = every prefix's descent set correct)
train_seq_acc = getPredictions(model, train_tokens, train_targets, train_mask)

for i in range(len(train_tokens)):
    input_seq = train_tokens[i]
    word_str  = ' '.join([str(int(tok)) for tok in input_seq.tolist()])
    word_len  = (input_seq != 0).sum().item()   # no special token in the new format

    words.append(word_str)
    seq_accuracies.append(train_seq_acc[i].item())
    lengths.append(word_len)
    dataset_indices.append(i)

    total_by_length[word_len] += 1
    if train_seq_acc[i].item() < 1.0:
        misclassified_by_length[word_len] += 1

In [ ]:
print(misclassified_by_length)

In [ ]:
# Collect imperfect sequences (seq accuracy < 1.0)
imperfect = [
    (words[i], lengths[i], seq_accuracies[i], dataset_indices[i])
    for i in range(len(words)) if seq_accuracies[i] < 1.0
]

print("\nRandom Sample of Imperfect Train Sequences:")
for word, length, acc, idx in random.sample(imperfect, min(20, len(imperfect))):
    print(f"Word: {word} | Length: {length} | Seq Acc: {acc:.3f} | Index: {idx}")

# Plot 1: Raw length distribution of imperfect sequences
imperfect_lengths = [length for _, length, _, _ in imperfect]
plt.figure(figsize=(10, 6))
plt.hist(imperfect_lengths, bins=range(min(imperfect_lengths), max(imperfect_lengths)+2), edgecolor='black')
plt.title("Length Distribution of Imperfect Train Sequences (Post Training)")
plt.xlabel("Word Length")
plt.ylabel("Number of Sequences")
plt.grid(True)
plt.tight_layout()
plt.show()

# Plot 2: Percentage imperfect per length
lengths_sorted        = sorted(total_by_length.keys())
percent_misclassified = [100 * misclassified_by_length[l] / total_by_length[l] for l in lengths_sorted]
misclassified_counts  = [misclassified_by_length[l] for l in lengths_sorted]

plt.figure(figsize=(10, 6))
bars = plt.bar(lengths_sorted, percent_misclassified, edgecolor='black')

for bar, count in zip(bars, misclassified_counts):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, height + 1, f'{count}',
             ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.title("Percentage of Imperfect Sequences by Length (Train)")
plt.xlabel("Word Length")
plt.ylabel("Imperfect Rate (%)")
plt.xticks(lengths_sorted)
plt.grid(True)
plt.tight_layout()
plt.show()

### Test Misclassification Graphs

In [ ]:
words = []
seq_accuracies = []
lengths = []
dataset_indices = []

total_by_length         = defaultdict(int)
misclassified_by_length = defaultdict(int)

test_tokens   = test_tokens.to(device1)
test_seq_acc  = getPredictions(model, test_tokens, test_targets, test_mask)

for i in range(len(test_tokens)):
    input_seq = test_tokens[i]
    word_str  = ' '.join([str(int(tok)) for tok in input_seq.tolist()])
    word_len  = (input_seq != 0).sum().item()   # no special token in the new format

    words.append(word_str)
    seq_accuracies.append(test_seq_acc[i].item())
    lengths.append(word_len)
    dataset_indices.append(i)

    total_by_length[word_len] += 1
    if test_seq_acc[i].item() < 1.0:
        misclassified_by_length[word_len] += 1

imperfect = [
    (words[i], lengths[i], seq_accuracies[i], dataset_indices[i])
    for i in range(len(words)) if seq_accuracies[i] < 1.0
]

print("\nRandom Sample of Imperfect Test Sequences:")
for word, length, acc, idx in random.sample(imperfect, min(20, len(imperfect))):
    print(f"Word: {word} | Length: {length} | Seq Acc: {acc:.3f} | Index: {idx}")

# Plot 1: Raw length distribution of imperfect sequences
imperfect_lengths = [length for _, length, _, _ in imperfect]
plt.figure(figsize=(10, 6))
plt.hist(imperfect_lengths, bins=range(min(imperfect_lengths), max(imperfect_lengths)+2), edgecolor='black')
plt.title("Length Distribution of Imperfect Test Sequences (Post Training)")
plt.xlabel("Word Length")
plt.ylabel("Number of Sequences")
plt.grid(True)
plt.tight_layout()
plt.show()

# Plot 2: Percentage imperfect per length
lengths_sorted        = sorted(total_by_length.keys())
percent_misclassified = [100 * misclassified_by_length[l] / total_by_length[l] for l in lengths_sorted]
misclassified_counts  = [misclassified_by_length[l] for l in lengths_sorted]

plt.figure(figsize=(10, 6))
bars = plt.bar(lengths_sorted, percent_misclassified, edgecolor='black')

for bar, count in zip(bars, misclassified_counts):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, height + 1, f'{count}',
             ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.title("Percentage of Imperfect Sequences by Length (Test)")
plt.xlabel("Word Length")
plt.ylabel("Imperfect Rate (%)")
plt.xticks(lengths_sorted)
plt.grid(True)
plt.tight_layout()
plt.show()

### Other Graphs
- check on misclassified
- attention heads of random selection of misclassified train inputs

In [ ]:
print(f"total word count: {sum(total_by_length.values())}")
print(f"total misclassified count: {sum(misclassified_by_length.values())}")

In [ ]:
# Prints 1 graph per attention head
# x: letters (keys attended to)
# y: letters (query giving attention to x axis)

# Make sure there are at least 4 items to sample from
sample_size = min(4, len(misclassified))

# Take random sample of misclassified items
sample = random.sample(misclassified, sample_size)

misclassified_index = [index for _, _, _, index in sample]
# Move to test device, because it has more memory
for mis_index in misclassified_index:
    chosenWord = test_tokens[mis_index].unsqueeze(0).to(device1)  # shape (1, 22)


    # Generate labeled tokens with position

    # ex: word len 22: str_tokens = [0,...,21]
    str_tokens = [f"{i}" for i, tok in enumerate(chosenWord[0])]

    # Visualize
    n_heads = model.cfg.n_heads
    n_layers = model.cfg.n_layers
    for layer in range(n_layers):
        for head in range(n_heads):
            # gets the attention pattern for layer 0, pick 1 word from the batch and look at 1 head for it
            # detatch tensor from computation graph (no gradients), move tensor to cpu to make imshow heatmap
            attn_single_head = cache["pattern", layer][mis_index, head].detach().cpu()
            
            imshow(
                attn_single_head,
                x=str_tokens,
                y=str_tokens,
                xaxis="Key (Attended To)",
                yaxis="Query (Paying Attention)",
                title=f"Layer {layer} Head {head} Attention Pattern (for word {mis_index})(Positional Tokens)"
            )